# AllocRL — Colab 학습 노트북

CNN 공간 인식 + 블록-집합 attention(미래 lookahead) 기반 블록 배치 강화학습.

**사용법:** 위에서부터 셀을 실행하되, 하이퍼파라미터는 **셀 4만 수정**하고 **셀 5를 다시 실행**하면 바로 반영됩니다.

주요 신규 옵션
- `--extractor block-attn` : 블록 토큰 집합 self-attention 추출기
- `--n-future-blocks N` : 미래 블록 관측 수 (0=미사용, block-attn엔 3~5 권장)

⚠️ 기존 CNN 체크포인트는 관측/네트워크가 달라 block-attn으로 이어학습(resume) 불가 — 처음부터 학습하세요.


### 1. 최신 코드 클론 (재실행해도 항상 최신)

In [ ]:
%cd /content
import os, shutil
if os.path.exists('/content/CNN-RL'):
    shutil.rmtree('/content/CNN-RL')      # 재실행 시 최신 반영 위해 재클론
!git clone https://github.com/LMS4681/CNN-RL.git
%cd /content/CNN-RL/AllocRL
!ls

### 2. 의존성 설치

In [ ]:
# Colab엔 torch/numpy/pandas/matplotlib/tensorboard가 이미 있음.
# gymnasium/sb3/sb3-contrib/onnx만 추가 설치되며 torch는 재설치되지 않음.
!pip install -q -r requirements.txt
# ↑ "Restart runtime" 안내가 뜨면: 런타임 재시작 후 셀 1부터 다시 실행.

### 3. (선택) 빠른 검증

In [ ]:
# 긴 학습 전 관측/네트워크 정상 동작 점검 (선택)
!python test_future_block_lookahead.py            # 순수 로직 (의존성 불필요)
!python test_block_attention_and_future_obs.py    # 미래 관측 shape + block-attn 추출기
!python smoke_test.py                             # 환경 전체 스모크

### 4. 하이퍼파라미터 ⭐ (이 셀만 수정)

In [ ]:
# ===================== 여기만 수정 =====================
# --- 모델 / 관측 구조 ---
EXTRACTOR       = "block-attn"   # "cnn" | "pointer-attn" | "block-attn"
N_FUTURE_BLOCKS = 4              # 미래 lookahead 블록 수 (0=미사용). block-attn엔 3~5 권장
EMBED_DIM       = 64             # attn 토큰 임베딩 차원 (pointer-attn / block-attn 전용)
NUM_HEADS       = 4              # attention head 수 (EMBED_DIM % NUM_HEADS == 0 이어야 함)
FEATURES_DIM    = 256
CNN_OUT_DIM     = 64
GRID_SIZE       = 64             # 점유 그리드 해상도 (메모리 영향)

# --- 학습 하이퍼파라미터 ---
TIMESTEPS   = 100_000
LR          = 3e-4
N_STEPS     = 554
BATCH_SIZE  = 64
N_EPOCHS    = 10
GAMMA       = 1.0

# --- 실행 환경 ---
N_ENVS      = 1                  # 병렬 환경 수 (1이면 단일, >1이면 VEC_ENV 사용)
VEC_ENV     = "auto"             # "auto" | "dummy" | "subproc"
DEVICE      = "auto"             # "auto"면 Colab GPU 자동 사용
N_EVAL      = 5
ACTIVE_WORKSPACE_CODES = "PE052,PE055,PE051,PE050,PE049"  # ""(빈문자열)=전체 작업장

# --- 입출력 ---
OUTPUT_DIR  = "./output_block_attn"
RESUME_FROM = ""                 # 이어학습할 .zip 경로 (없으면 ""). ※관측/추출기 동일할 때만!
EXPORT_ONNX = True
# =======================================================
print(f"설정: extractor={EXTRACTOR}, n_future_blocks={N_FUTURE_BLOCKS}, "
      f"timesteps={TIMESTEPS}, output={OUTPUT_DIR}")

### 5. 학습 실행

In [ ]:
# 셀 4의 값으로 학습 명령을 구성해 실행. 셀 4를 바꾸고 이 셀만 다시 실행하면 반영됩니다.
cmd = (
    "python train.py"
    " --data-dir ./data"
    f" --output-dir {OUTPUT_DIR}"
    f" --timesteps {TIMESTEPS}"
    f" --lr {LR}"
    f" --n-steps {N_STEPS}"
    f" --batch-size {BATCH_SIZE}"
    f" --n-epochs {N_EPOCHS}"
    f" --gamma {GAMMA}"
    f" --grid-size {GRID_SIZE}"
    f" --extractor {EXTRACTOR}"
    f" --n-future-blocks {N_FUTURE_BLOCKS}"
    f" --features-dim {FEATURES_DIM}"
    f" --cnn-out-dim {CNN_OUT_DIM}"
    f" --extractor-embed-dim {EMBED_DIM}"
    f" --extractor-heads {NUM_HEADS}"
    f" --n-envs {N_ENVS}"
    f" --vec-env {VEC_ENV}"
    f" --device {DEVICE}"
    f" --n-eval {N_EVAL}"
    f' --active-workspace-codes "{ACTIVE_WORKSPACE_CODES}"'
)
if RESUME_FROM:
    cmd += f" --resume-from {RESUME_FROM}"
if not EXPORT_ONNX:
    cmd += " --no-export-onnx"
print(cmd, "\n" + "="*70)
!{cmd}

### 6. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir .    # 모든 output* run이 함께 보임 → cnn vs block-attn A/B 비교

### 7. (선택) 배치 시각화

In [ ]:
# 배치 시각화 (선택). --n-future-blocks 를 학습 때 값과 반드시 동일하게 (관측 공간 일치)!
cmd = (
    "python visualize_eval_placement.py"
    f" --model-path {OUTPUT_DIR}/block_placement_ppo.zip"
    f" --output-dir {OUTPUT_DIR}/eval_visualization"
    f" --grid-size {GRID_SIZE}"
    f" --n-future-blocks {N_FUTURE_BLOCKS}"
    f' --active-workspace-codes "{ACTIVE_WORKSPACE_CODES}"'
)
print(cmd)
!{cmd}

### 8. (선택) Google Drive에 결과 저장

In [ ]:
# Colab은 세션 종료 시 파일이 사라짐 → 결과를 Drive에 저장 (선택)
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
dst = '/content/drive/MyDrive/CNN-RL-outputs/' + os.path.basename(OUTPUT_DIR)
shutil.copytree(OUTPUT_DIR, dst, dirs_exist_ok=True)
print("저장됨:", dst)

## 팁

- **A/B 비교**: 셀 4에서 `(EXTRACTOR, N_FUTURE_BLOCKS, OUTPUT_DIR)`만 바꿔 여러 번 학습하면
  (`output_cnn`, `output_block_attn` 등) 셀 6 TensorBoard에서 한눈에 비교됩니다.
  예: `("cnn", 0)` vs `("block-attn", 3)` vs `("block-attn", 5)`.
- **하이퍼파라미터 즉시 반영**: 셀 4 수정 → 셀 5 재실행. 다른 셀은 다시 안 돌려도 됩니다.
- `EMBED_DIM % NUM_HEADS == 0` 이어야 attn 추출기가 생성됩니다.
- `block-attn` + `N_FUTURE_BLOCKS=0`이면 경고 출력(단일 토큰 → MLP와 유사하게 퇴화).
